In [156]:
import boto3
from dotenv import load_dotenv
import os
import psycopg2

load_dotenv(verbose=True)

import sys
# sys.path에 상위 폴더(..)를 추가합니다.
sys.path.append("..")

* subscription_data -> pipeline_queue_tb

In [162]:
# --- 데이터베이스 연결 설정 ---
# django_test 데이터베이스 연결 정보
DB_DJANGO_HOST = os.getenv("DB_HOST")
DB_DJANGO_NAME = "req_sub_db"
DB_DJANGO_USER = os.getenv("DB_USER")
DB_DJANGO_PASSWORD = os.getenv("DB_PASSWORD")
DB_DJANGO_PORT = "5432" # 기본 PostgreSQL 포트

# mart_db 데이터베이스 연결 정보
DB_MART_HOST = os.getenv("DB_HOST")
DB_MART_NAME = "mart_db"
DB_MART_USER = os.getenv("DB_USER")
DB_MART_PASSWORD = os.getenv("DB_PASSWORD")
DB_MART_PORT = "5432" # 기본 PostgreSQL 포트

# 원본 테이블 이름 (django_test DB에 있는 테이블)
SOURCE_TABLE_NAME = "subscription_data" # 예: 'subscription_data'
# 대상 테이블 이름 (mart_db DB에 있는 테이블)
DESTINATION_TABLE_NAME = "pipeline_queue_tb"

django_conn = None
mart_conn = None
# 1. django_test 데이터베이스에 연결
print(f"{DB_DJANGO_NAME} 데이터베이스에 연결 중...")
django_conn = psycopg2.connect(
    host=DB_DJANGO_HOST,
    database=DB_DJANGO_NAME,
    user=DB_DJANGO_USER,
    password=DB_DJANGO_PASSWORD,
    port=DB_DJANGO_PORT,
)
django_cursor = django_conn.cursor()
print(f"{DB_DJANGO_NAME} 데이터베이스 연결 성공.")

# 2. mart_db 데이터베이스에 연결
print(f"{DB_MART_NAME} 데이터베이스에 연결 중...")
mart_conn = psycopg2.connect(
    host=DB_MART_HOST,
    database=DB_MART_NAME,
    user=DB_MART_USER,
    password=DB_MART_PASSWORD,
    port=DB_MART_PORT
)
mart_cursor = mart_conn.cursor()
print(f"{DB_MART_NAME} 데이터베이스 연결 성공.")

# 3. django_test에서 데이터 읽기
print(f"'{SOURCE_TABLE_NAME}' 테이블에서 데이터 읽기...")
# 원본 테이블의 모든 컬럼을 선택합니다.
django_cursor.execute(f"SELECT * FROM {SOURCE_TABLE_NAME};")
rows = django_cursor.fetchall()
print(f"{len(rows)}개의 행을 읽었습니다.")

if not rows:
    print("복사할 데이터가 없습니다.")

# 4. 대상 테이블의 컬럼 정보 가져오기 (INSERT 및 UPDATE 문 생성을 위함)
mart_cursor.execute(f"SELECT column_name FROM information_schema.columns WHERE table_schema = 'public' AND table_name = '{DESTINATION_TABLE_NAME}' ORDER BY ordinal_position;")
destination_columns = [col[0] for col in mart_cursor.fetchall()]

# 원본 테이블의 컬럼 정보 가져오기 (데이터 매핑을 위함)
django_cursor.execute(f"SELECT column_name FROM information_schema.columns WHERE table_schema = 'public' AND table_name = '{SOURCE_TABLE_NAME}' ORDER BY ordinal_position;")
source_columns = [col[0] for col in django_cursor.fetchall()]

# 'exp_id' 컬럼이 UPSERT의 기준이 되므로, 양쪽 테이블에 존재하는지 확인합니다.
if 'exp_id' not in source_columns or 'exp_id' not in destination_columns:
    print("오류: UPSERT를 위해 'exp_id' 컬럼이 원본 또는 대상 테이블에 필수적입니다.")

# 삽입/업데이트를 위해 데이터를 대상 테이블의 컬럼 순서에 맞게 매핑합니다.
mapped_rows = []
for row_data in rows:
    source_row_dict = dict(zip(source_columns, row_data))
    # 대상 테이블의 컬럼 순서에 맞춰 데이터를 재구성합니다.
    dest_row_tuple = tuple(source_row_dict.get(col) for col in destination_columns)
    mapped_rows.append(dest_row_tuple)

# UPSERT 쿼리 생성
# 대상 테이블의 'exp_id' 컬럼에 UNIQUE 제약 조건 또는 PRIMARY KEY가 설정되어 있어야 합니다.
# 만약 설정되어 있지 않다면, 'ALTER TABLE public.pipeline_queue_tb ADD CONSTRAINT unique_exp_id UNIQUE (exp_id);' 와 같이 추가해야 합니다.

# INSERT 부분에 사용될 컬럼 목록과 플레이스홀더
insert_cols_str = ', '.join(destination_columns)
placeholders = ', '.join(['%s'] * len(destination_columns))

# DO UPDATE SET 부분에 사용될 컬럼 목록 (exp_id 제외)
# 이 컬럼들이 변경되었을 때만 업데이트가 일어납니다.
excluded_col = ['exp_id', 'test_file_upload_request_state', 'test_end_datetime', 'last_chnn_end_datetime', 'test_file_last_upload_datetime']
update_cols = [col for col in destination_columns if col not in excluded_col]
set_clause = ', '.join([f"{col} = EXCLUDED.{col}" for col in update_cols])

# 특정 컬럼이 바뀌었을 때만 업데이트되도록 WHERE 절 추가
# 'target'은 현재 테이블에 존재하는 행을, 'EXCLUDED'는 삽입하려는 새 행을 나타냅니다.
# 'IS DISTINCT FROM'은 NULL 값을 안전하게 비교하여 값이 다른 경우에만 TRUE를 반환합니다.
where_clause_parts = []
for col in update_cols:
    where_clause_parts.append(f"{DESTINATION_TABLE_NAME}.{col} IS DISTINCT FROM EXCLUDED.{col}")

where_clause = ""
if where_clause_parts:
    where_clause = f" WHERE {' OR '.join(where_clause_parts)}"

upsert_query = f"""
    INSERT INTO public.{DESTINATION_TABLE_NAME} ({insert_cols_str})
    VALUES ({placeholders})
    ON CONFLICT (exp_id) DO UPDATE SET
        {set_clause}
    {where_clause};
"""

# 5. mart_db에 데이터 삽입/업데이트
print(f"'{DESTINATION_TABLE_NAME}' 테이블에 데이터 삽입/업데이트 중...")
# executemany를 사용하여 여러 행을 한 번에 처리합니다.
mart_cursor.executemany(upsert_query, mapped_rows)
mart_conn.commit() # 변경사항 커밋
print(f"'{DESTINATION_TABLE_NAME}' 테이블에 {mart_cursor.rowcount}개의 행이 성공적으로 처리되었습니다 (삽입 또는 업데이트).")

req_sub_db 데이터베이스에 연결 중...
req_sub_db 데이터베이스 연결 성공.
mart_db 데이터베이스에 연결 중...
mart_db 데이터베이스 연결 성공.
'subscription_data' 테이블에서 데이터 읽기...
29개의 행을 읽었습니다.
'pipeline_queue_tb' 테이블에 데이터 삽입/업데이트 중...
'pipeline_queue_tb' 테이블에 0개의 행이 성공적으로 처리되었습니다 (삽입 또는 업데이트).


### mart db 에서 path 추출 후 raw 다운로드 to temp

* download_required_yn이 null or True 이면 다운로드 후 True or False 변경
* download_required_yn이 False 이면 pass
* 모든 행 다운로드 후에는 로직 검증 후 download_requried_yn을 변경.

In [9]:
from src.voltaflow.connectors.db import DbConnector

host = os.getenv("DB_HOST")
database = "mart_db"
user = os.getenv("DB_USER")
password = os.getenv("DB_PASSWORD")
port = os.getenv("DB_PORT")

db_conn = DbConnector(host, database, user, password, port)
db_conn.connect()
DESTINATION_TABLE_NAME = "pipeline_queue_tb"
query = f"SELECT * FROM {DESTINATION_TABLE_NAME} WHERE download_required_yn IS NULL OR TRUE;"
temp_db = db_conn.fetch_data(query ,as_dataframe=True)

데이터베이스 'mart_db'에 성공적으로 연결되었습니다.


In [150]:
import requests
idx = 13
temp_folder = r"C:\Users\jhchoei\Desktop\Workspace\VoltaFlow\tmp"
target_path = os.path.join(temp_folder, temp_db["file_full_path_cts"][idx].split("/")[-1])

with requests.get(temp_db["file_full_path_cts"][idx]) as res:
    if res.status_code == 200:
        with open(target_path, "wb") as f:
            f.write(res.content)
        print("파일 다운로드 성공")
    else:
        print(f"파일 다운로드 실패: {res.status_code}")

파일 다운로드 성공


### binary file to parquet, Minio에 업로드

In [151]:
from src.voltaflow.parsers.Factory import FactoryProcessor
from assets import config
from src.voltaflow.connectors.minio import MinioConnector
import botocore
import io
import os

In [152]:
processor = FactoryProcessor.create_processor(str(target_path), config)
stepend_df = processor.parse_binary_file(target_path)['StepEndData']

In [153]:
pq_buffer = io.BytesIO()
stepend_df.to_parquet(pq_buffer, index=False, compression='snappy')
pq_buffer.seek(0)
pq_size = pq_buffer.getbuffer().nbytes

In [154]:
file_name = temp_db['cell_id'][idx] + "/" + \
            temp_db['exp_id'][idx] + "/" + \
            target_path.split("\\")[-1].replace(".cts", ".parquet")

In [155]:
try:    
    minio_connector = MinioConnector()
    res = minio_connector.client.put_object(Bucket='tos-stepend',
                                            Key=file_name,
                                            Body=pq_buffer,
                                            ContentLength=pq_size,
                                            ContentType='application/x-parquet'
                                            )
    
except botocore.exceptions.ConnectTimeoutError as e:
    print(f"ConnectTimeoutError: {e}")

else:
    os.remove(target_path)

Minio connection established successfully
